# Multi-Agent · Day 39 — **Agent Memory Architecture**

**Phase 2 · Module 6: Agent Memory, Tools & MCP** · ~2.5 hrs

An agent with no memory re-meets every customer as a stranger. Real agentic systems have a **memory architecture** — and getting the taxonomy right is what separates "I stuffed everything in the prompt" from an AVP-level design. There are **four types**, and they differ by *lifetime* and *scope*:

| Type | Lives | Scope | Banking example |
|---|---|---|---|
| **In-context** (working) | one turn | this request | the messages in the current prompt |
| **Episodic** | across sessions | this customer | "last call they disputed a £48 fee" |
| **Semantic** | long-term | shared/global | "SME overdraft policy = 20% of turnover" |
| **Procedural** | long-term | the agent itself | "always screen sanctions before quoting" |

Today we build episodic memory for real (store → retrieve past interactions) and use LangGraph's checkpointer as the **cross-session** substrate so an agent remembers a customer's preferences. Runnable, no external DB. (Today's Python notebook implements the episodic retrieval scoring from scratch.)

Today:
1. The **four memory types** as code — what each actually is.
2. **Episodic memory**: store + retrieve past interactions, recency-weighted.
3. **Cross-session recall** with LangGraph `MemorySaver` + a `thread_id` per customer.
4. **Writing memories back** — the consolidation loop that keeps memory useful.

## 1. The four types, made concrete

The mistake is treating "memory" as one bucket. Each type has a different store, lifetime, and retrieval rule. Naming them separately is what lets you reason about *what to persist where*.

In [1]:
from dataclasses import dataclass, field
import time

# 1. IN-CONTEXT — ephemeral, dies with the request. Just the current messages.
working_memory = [{"role": "user", "content": "What's my overdraft limit?"}]

# 2. EPISODIC — this customer's past events, across sessions.
@dataclass
class Episode:
    customer_id: str
    text: str
    ts: float = field(default_factory=time.time)

# 3. SEMANTIC — global facts/policy, not tied to a customer.
semantic_memory = {
    "sme_overdraft_policy": "Overdraft limit = 20% of annual turnover.",
    "sanctions_rule": "Any OFAC/HMT hit blocks the facility pending review.",
}

# 4. PROCEDURAL — how the agent behaves; its learned/authored routines.
procedural_memory = [
    "Always screen the counterparty for sanctions before quoting a limit.",
    "If confidence < 0.6, refer to a human instead of auto-approving.",
]

print("in-context turns :", len(working_memory))
print("semantic keys    :", list(semantic_memory))
print("procedural rules :", len(procedural_memory))

in-context turns : 1
semantic keys    : ['sme_overdraft_policy', 'sanctions_rule']
procedural rules : 2


☝ The value of the taxonomy is **routing writes to the right store**. A customer's fee dispute is *episodic* (their scope, persists) — it must not leak into *semantic* memory (global, shared) or you'd apply one customer's history to everyone. A policy change is *semantic*. "Always screen sanctions first" is *procedural* — it changes the agent's behaviour, not its facts. Mixing these is the root cause of agents that either forget everything or over-remember and hallucinate cross-customer.

## 2. Episodic memory — store + retrieve

Episodic memory answers "what happened before with *this* customer?" Store events; retrieve the most relevant, with a **recency bias** so a dispute from last week outranks a routine query from last year. (Real systems retrieve by embedding similarity; here we keep it deterministic with keyword overlap + recency — the Python notebook builds the similarity version.)

In [2]:
class EpisodicMemory:
    def __init__(self):
        self._episodes: list[Episode] = []

    def store(self, customer_id: str, text: str, ts: float | None = None) -> None:
        self._episodes.append(Episode(customer_id, text, ts or time.time()))

    def retrieve(self, customer_id: str, query: str, top_k: int = 3,
                 now: float | None = None) -> list[str]:
        now = now or time.time()
        q = set(query.lower().split())
        scored = []
        for e in self._episodes:
            if e.customer_id != customer_id:                 # scope: this customer only
                continue
            overlap = len(q & set(e.text.lower().split()))
            age_days = (now - e.ts) / 86400
            recency = 1 / (1 + age_days)                     # newer -> closer to 1
            score = overlap + 0.5 * recency                  # relevance + recency bias
            scored.append((score, e.text))
        scored.sort(reverse=True)
        return [text for _, text in scored[:top_k]]

epi = EpisodicMemory()
DAY = 86400; now = time.time()
epi.store("C-4417", "disputed a 48 GBP overdraft fee, refunded", ts=now - 5*DAY)
epi.store("C-4417", "asked about increasing overdraft limit", ts=now - 200*DAY)
epi.store("C-4417", "routine balance check", ts=now - 1*DAY)
epi.store("C-9001", "flagged for sanctions review", ts=now - 2*DAY)   # different customer

print("recall for C-4417 re 'overdraft fee':")
for r in epi.retrieve("C-4417", "overdraft fee dispute", now=now):
    print("  -", r)

recall for C-4417 re 'overdraft fee':
  - disputed a 48 GBP overdraft fee, refunded
  - asked about increasing overdraft limit
  - routine balance check


☝ Two rules do the heavy lifting. **Scope filter** (`e.customer_id != customer_id`) — C-9001's sanctions flag can *never* surface for C-4417; that isolation is a compliance requirement, not a nicety. **Recency bias** (`1/(1+age_days)`) — the 200-day-old limit query loses to the 5-day-old fee dispute even on similar keywords, because in banking recent context usually matters more. Retrieval is where memory becomes useful or dangerous: retrieve the wrong customer's episode into a prompt and you've built a data-leak machine.

## 3. Cross-session recall with LangGraph

The point of episodic memory is *persistence across sessions*. LangGraph's checkpointer gives an agent durable state keyed by `thread_id` — use **one thread per customer** and the agent picks up exactly where it left off, days later.

In [3]:
from typing import Annotated, TypedDict
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

class ConvState(TypedDict):
    turns: Annotated[list[str], operator.add]      # append-only across sessions
    preference: str                                # remembered customer preference

def agent_turn(state: ConvState) -> dict:
    # In session 1 the customer states a preference; later sessions read it back.
    latest = state["turns"][-1] if state["turns"] else ""
    updates = {}
    if "prefer" in latest.lower():
        updates["preference"] = latest.split("prefer", 1)[1].strip(" .")
    return updates

g = StateGraph(ConvState)
g.add_node("agent", agent_turn)
g.add_edge(START, "agent"); g.add_edge("agent", END)
agent = g.compile(checkpointer=MemorySaver())          # durable per thread_id

cust = {"configurable": {"thread_id": "customer:C-4417"}}   # one thread per customer

# --- Session 1 (today): customer states a preference ---
agent.invoke({"turns": ["I prefer email contact, not phone"]}, cust)
# --- Session 2 (days later, brand new invocation): agent still remembers ---
s2 = agent.invoke({"turns": ["Any update on my limit?"]}, cust)
print("remembered preference :", s2["preference"])
print("full history in thread:", s2["turns"])

remembered preference : email contact, not phone
full history in thread: ['I prefer email contact, not phone', 'Any update on my limit?']


☝ Session 2 is a *fresh* `invoke` with no mention of the preference — yet the agent recalls "email contact" because the checkpointer rehydrated the thread's state by `thread_id`. That's the mechanism behind "the agent remembers me": **thread = customer**, checkpointer = the memory substrate. Swap `MemorySaver` for `PostgresSaver` and the memory survives process restarts and scales across a cluster — same code. This is *in-context memory made durable*: the graph reloads prior turns into working memory automatically.

## 4. Writing memories back — the consolidation loop

Memory that only grows becomes noise. The mature pattern: after a session, **consolidate** — promote durable facts to episodic/semantic storage and summarise the rest, so retrieval stays sharp.

In [4]:
def consolidate(session_turns: list[str], customer_id: str,
                epi: EpisodicMemory, semantic: dict) -> dict:
    """End-of-session: route each turn to the right long-term store."""
    written = {"episodic": 0, "semantic": 0, "dropped": 0}
    for turn in session_turns:
        low = turn.lower()
        if "prefer" in low or "disputed" in low or "requested" in low:
            epi.store(customer_id, turn)                 # customer event -> episodic
            written["episodic"] += 1
        elif "policy" in low or "rule" in low:
            semantic[f"note_{len(semantic)}"] = turn     # global fact -> semantic
            written["semantic"] += 1
        else:
            written["dropped"] += 1                       # small talk -> forget
    return written

turns = ["I prefer email contact", "what's the weather",
         "disputed a duplicate charge", "policy: waive first late fee for premier tier"]
report = consolidate(turns, "C-4417", epi, semantic_memory)
print("consolidation:", report)
print("episodic now has C-4417 events:",
      len(epi.retrieve("C-4417", "prefer disputed", top_k=10)))

consolidation: {'episodic': 2, 'semantic': 1, 'dropped': 1}
episodic now has C-4417 events: 5


☝ Consolidation is the difference between memory that *helps* and memory that *rots*. Not everything is worth keeping — "what's the weather" is dropped; a preference and a dispute go to **episodic** (customer scope); a policy note goes to **semantic** (global). This routing is exactly the taxonomy from cell 1 applied as a *write policy*. Production systems (Mem0, Zep) run this loop with an LLM deciding what's memory-worthy and periodically summarising old episodes into compact facts — but the routing logic is what you must be able to design.

## 5. Recap

| Memory type | Store here | Retrieval rule | Cell |
|---|---|---|---|
| In-context (working) | the prompt / graph state | all of it, this turn | 1, 3 |
| Episodic | per-customer event log | scope + relevance + **recency** | 2 |
| Semantic | global KV / vector store | similarity, customer-agnostic | 1, 4 |
| Procedural | agent's routines | applied as behaviour | 1 |
| Cross-session persistence | checkpointer, thread=customer | rehydrate by `thread_id` | 3 |
| Keep memory sharp | consolidation loop | route + summarise + drop | 4 |

**The one-liner:** *scope and lifetime decide the store; recency and relevance decide retrieval; consolidation keeps it from rotting.* Tomorrow (Day 40): **semantic memory** backed by a real vector store — embeddings, similarity search, and FAISS vs pgvector for agent recall.